# Computer-Use Agents: Screens as Observations, the Mouse as a Tool

The runnable code of [Computer-Use Agents: Screens as Observations, the Mouse as a Tool](https://residual-stream.lvtan71.workers.dev/posts/computer-use-agents/), in the order the post runs it. Each
code cell is followed by the output the post printed, so you can compare.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lvtan71/residual-stream-code/blob/main/notebooks/computer-use-agents.ipynb)

**Hardware:** Runs on a CPU; Colab's free runtime is enough. About 1 min end to end.
The accessibility trees were snapshots of three live pages (this blog's home page, Hacker News and Wikipedia), captured once with Chrome's devtools; the files were not kept, and the live pages have changed since, so this notebook skips those three lines. The screenshot token counts run as in the post.

The post's numbers come from an Apple M4 laptop. Other hardware and library
versions can change the last digits of timings and of some floating-point
results; the conclusions should not change.

In [ ]:
# Setup: fetch the helpers and data this post's code uses into a working folder.
import os, shutil, subprocess, sys
SLUG = "computer-use-agents"
ROOT = os.path.abspath("..") if os.path.isdir("../shared") else os.path.abspath("residual-stream-code")
if not os.path.isdir(ROOT):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/lvtan71/residual-stream-code", ROOT], check=True)
WORK = os.path.join(ROOT, "work", SLUG)
os.makedirs(WORK, exist_ok=True)
for f in []:
    shutil.copy(os.path.join(ROOT, f), WORK)
    if f.endswith(".tar.gz"):                       # the blog's posts, as of that run
        shutil.unpack_archive(os.path.join(WORK, os.path.basename(f)), WORK)
os.chdir(WORK)
sys.path.insert(0, WORK)
print("working in", WORK)

## Screenshots or the accessibility tree

[Read this part in the post](https://residual-stream.lvtan71.workers.dev/posts/computer-use-agents/#screenshots-or-the-accessibility-tree)

In [ ]:
import math
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download

# What a computer-use agent's observations cost. Screenshots: Claude's
# documented rule, one visual token per 28x28 patch, after shrinking the
# image (keeping its shape) to fit a tier's long-edge and token limits.
TIERS = {"standard": (1568, 1568), "high-resolution": (2576, 4784)}     # (max long edge, max visual tokens)
def visual_tokens(w, h, tier):
    edge, cap = TIERS[tier]
    tokens = lambda w, h: math.ceil(w / 28) * math.ceil(h / 28)
    if max(w, h) <= edge and tokens(w, h) <= cap:
        return w, h, tokens(w, h)
    lo, hi = 0.0, min(1.0, edge / max(w, h))            # largest scale that fits both limits
    for _ in range(60):
        mid = (lo + hi) / 2
        (lo, hi) = (mid, hi) if tokens(round(w * mid), round(h * mid)) <= cap else (lo, mid)
    return round(w * lo), round(h * lo), tokens(round(w * lo), round(h * lo))

print("screenshot tokens (check against the documentation's table: 1920x1080 -> 1456x819, 1560 / 2691)")
for w, h in ((1024, 768), (1280, 800), (1920, 1080), (2560, 1440)):
    row = [f"{w}x{h}"]
    for tier in TIERS:
        sw, sh, t = visual_tokens(w, h, tier)
        row.append(f"{tier}: {t:5,d} tokens" + ("" if (sw, sh) == (w, h) else f" (shrunk to {sw}x{sh})"))
    print("  " + "   ".join(row))

# Accessibility trees: the same pages as text, captured with Chrome's
# devtools at 1280x800 and counted with Qwen2.5's tokenizer (an approximate
# stand-in for any model's). A raw snapshot covers the whole page, with every
# node, URL and identifier; agents usually get a pruned version, here only
# the interactive elements with their labels, URLs dropped.
tok = AutoTokenizer.from_pretrained(snapshot_download("Qwen/Qwen2.5-1.5B-Instruct",
                                                      revision="989aa7980e4cf806f80c7fef2b1adb7bc71aa306"))
import re
INTERACTIVE = ("link", "button", "textbox", "combobox", "checkbox", "radio", "searchbox", "menuitem", "tab", "option")
def pruned(text):
    keep = [l for l in text.splitlines() if l.strip().split(" ")[1:2] and l.strip().split(" ")[1] in INTERACTIVE]
    return "\n".join(re.sub(r' url="[^"]*"', "", l.strip()) for l in keep)
print("\naccessibility trees, whole page (raw devtools snapshot / interactive elements only, no URLs):")
for name in [n for n in ("blog_home", "hackernews", "wikipedia") if __import__("os").path.exists(f"a11y_{n}.txt")]:
    text = open(f"a11y_{name}.txt").read()
    p = pruned(text)
    print(f"  {name:11s} {len(text.splitlines()):6,d} nodes {len(tok(text).input_ids):9,d} tokens   "
          f"pruned: {len(p.splitlines()):5,d} elements {len(tok(p).input_ids):8,d} tokens")

# A task's input tokens. Each step the agent sees the newest screenshot
# plus the conversation so far; history grows unless old screenshots are
# dropped. 2,000 tokens of instructions and tools, 150 tokens of the
# model's own reasoning and action per step; output tokens not counted.
# With prompt caching ("Prompt Caching") a request reads the previous request's
# prefix at 0.1x the price and writes what's new at 1.25x. Dropping old
# screenshots changes the prefix, so a trimmed history can't be cached.
def task_tokens(steps, shot, keep=None):
    total = 0
    for t in range(1, steps + 1):
        shots = t if keep is None else min(t, keep)
        total += 2000 + 150 * t + shots * shot
    return total
def cached_cost(steps, shot, price=5e-6):
    cost, prev = 0.0, 0
    for t in range(1, steps + 1):
        now = 2000 + (150 + shot) * t
        cost += (0.1 * prev + 1.25 * (now - prev)) * price
        prev = now
    return cost
shot = visual_tokens(1280, 800, "standard")[2]
print(f"\none task's input tokens, screenshots at 1280x800 ({shot:,} tokens each), $5 per million input tokens:")
for steps in (10, 30, 50):
    a, b = task_tokens(steps, shot), task_tokens(steps, shot, keep=3)
    print(f"  {steps:3d} steps: keep every screenshot {a:>9,d} tokens (${a * 5e-6:5.2f}, cached ${cached_cost(steps, shot):5.2f}); "
          f"keep the last 3 {b:>8,d} (${b * 5e-6:5.2f}); last request {2000 + (150 + shot) * steps:,} tokens")

The post printed:

```text
screenshot tokens (check against the documentation's table: 1920x1080 -> 1456x819, 1560 / 2691)
  1024x768   standard: 1,036 tokens   high-resolution: 1,036 tokens
  1280x800   standard: 1,334 tokens   high-resolution: 1,334 tokens
  1920x1080   standard: 1,560 tokens (shrunk to 1456x819)   high-resolution: 2,691 tokens
  2560x1440   standard: 1,560 tokens (shrunk to 1456x819)   high-resolution: 4,784 tokens

accessibility trees, whole page (raw devtools snapshot / interactive elements only, no URLs):
  blog_home      109 nodes     3,748 tokens   pruned:    31 elements      557 tokens
  hackernews     756 nodes    15,685 tokens   pruned:   231 elements    3,090 tokens
  wikipedia    6,813 nodes   167,990 tokens   pruned: 1,150 elements   20,363 tokens

one task's input tokens, screenshots at 1280x800 (1,334 tokens each), $5 per million input tokens:
   10 steps: keep every screenshot   101,620 tokens ($ 0.51, cached $ 0.15); keep the last 3   64,268 ($ 0.32); last request 16,840 tokens
   30 steps: keep every screenshot   750,060 tokens ($ 3.75, cached $ 0.64); keep the last 3  245,808 ($ 1.23); last request 46,520 tokens
   50 steps: keep every screenshot 1,992,100 tokens ($ 9.96, cached $ 1.43); keep the last 3  487,348 ($ 2.44); last request 76,200 tokens
```